# Capstone 1 — Multi-Agent Research Assistant

**Book:** [Agentic AI: Build, Ship, Portfolio](../index.html)

---

In this capstone you will build a **multi-agent research system** that can:

1. Accept a research question from a user.
2. Search the web for relevant sources.
3. Extract and summarize content from each source.
4. Synthesize findings into a coherent narrative.
5. Manage citations in a structured format.
6. Generate a formatted research report.

A **Supervisor** agent orchestrates the entire pipeline, delegating to specialized sub-agents.

> **Note:** This notebook uses *mock/simulated* LLM responses so it runs without an API key. Replace the mock functions with real LLM calls (e.g., OpenAI, Anthropic) for production use.

---
## 1 Setup

In [ ]:
# --- Setup: Dependencies ---
# %pip install dataclasses-json

import json
import hashlib
import textwrap
import time
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime
from enum import Enum
from typing import Any

print("All imports ready.")

In [ ]:
# --- Mock LLM Helper ---
# Simulates an LLM call so the notebook runs without API keys.

def mock_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """Simulate an LLM response based on keywords in the prompt."""
    lower = (system_prompt + user_prompt).lower()
    if "search" in lower and "query" in lower:
        return json.dumps([
            {"title": "Recent Advances in Multi-Agent Systems",
             "url": "https://arxiv.org/abs/2401.00001",
             "snippet": "This paper surveys recent advances in multi-agent AI systems..."},
            {"title": "Collaborative AI Agents: A Framework",
             "url": "https://example.com/collab-agents",
             "snippet": "We propose a framework for collaborative AI agents that..."},
            {"title": "LLM-Powered Research Automation",
             "url": "https://example.com/llm-research",
             "snippet": "Large language models are increasingly used to automate..."}
        ])
    elif "extract" in lower or "summarize" in lower:
        return json.dumps({
            "summary": "The source discusses key developments in the field, highlighting "
                       "three main contributions: improved coordination protocols, "
                       "better task decomposition strategies, and novel evaluation metrics.",
            "key_facts": [
                "Multi-agent systems outperform single agents on complex tasks by 34%.",
                "Hierarchical delegation reduces token usage by 45%.",
                "Consensus-based synthesis improves factual accuracy."
            ],
            "relevance_score": 0.85
        })
    elif "synthe" in lower:
        return ("Based on the reviewed sources, multi-agent research systems show "
                "significant promise. Key findings include: (1) task decomposition "
                "is critical for quality, (2) hierarchical supervision reduces errors, "
                "and (3) structured citation management improves verifiability. "
                "However, challenges remain around latency and cost optimization.")
    elif "report" in lower or "format" in lower:
        return ("# Research Report\n\n## Executive Summary\n\n"
                "This report synthesizes findings from 3 sources on the topic.\n\n"
                "## Key Findings\n\n1. Multi-agent architectures improve research quality.\n"
                "2. Hierarchical supervision is essential.\n"
                "3. Citation management enhances trust.\n\n"
                "## Recommendations\n\nAdopt a supervisor-based multi-agent pattern.\n\n"
                "## References\n\n[See citation list]")
    else:
        return "Mock response: The system processed the request successfully."

print("Mock LLM helper ready.")

---
## 2 Web Search Agent

The **Search Agent** takes a research question, generates optimized search queries, and returns a ranked list of source URLs with snippets. In production this would call a search API (Google, Bing, Tavily, SerpAPI). Here we simulate it.

In [ ]:
@dataclass
class SearchResult:
    """A single web search result."""
    title: str
    url: str
    snippet: str
    query: str = ""
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class SearchAgent:
    """Agent that generates search queries and retrieves results."""

    def __init__(self, max_queries: int = 3):
        self.max_queries = max_queries
        self.search_history: list[SearchResult] = []

    def generate_queries(self, research_question: str) -> list[str]:
        """Use the LLM to decompose a research question into search queries."""
        # In production: call LLM to generate diverse queries
        base = research_question.lower().replace("?", "")
        queries = [
            f"{base} recent advances",
            f"{base} framework survey",
            f"{base} benchmarks evaluation",
        ]
        return queries[: self.max_queries]

    def search(self, query: str) -> list[SearchResult]:
        """Execute a single search query (mock)."""
        raw = mock_llm(
            system_prompt="You are a search engine. Return results as JSON.",
            user_prompt=f"Search query: {query}"
        )
        items = json.loads(raw)
        results = [SearchResult(title=r["title"], url=r["url"],
                                snippet=r["snippet"], query=query) for r in items]
        self.search_history.extend(results)
        return results

    def run(self, research_question: str) -> list[SearchResult]:
        """Full search pipeline: generate queries -> search -> deduplicate."""
        queries = self.generate_queries(research_question)
        all_results: list[SearchResult] = []
        seen_urls: set[str] = set()
        for q in queries:
            for r in self.search(q):
                if r.url not in seen_urls:
                    seen_urls.add(r.url)
                    all_results.append(r)
        print(f"[SearchAgent] {len(queries)} queries -> {len(all_results)} unique results")
        return all_results

print("SearchAgent defined.")

In [ ]:
# --- Test the Search Agent ---

search_agent = SearchAgent(max_queries=3)
results = search_agent.run("What are the latest advances in multi-agent AI systems?")

for i, r in enumerate(results, 1):
    print(f"  {i}. {r.title}")
    print(f"     {r.url}")
    print(f"     {r.snippet[:80]}...")
    print()

---
## 3 Content Extraction Agent

The **Extraction Agent** takes each search result, fetches the page content (mocked here), and extracts a structured summary with key facts and a relevance score.

In [ ]:
@dataclass
class ExtractedContent:
    """Structured content extracted from a source."""
    url: str
    title: str
    summary: str
    key_facts: list[str]
    relevance_score: float  # 0.0 to 1.0
    extraction_timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class ExtractionAgent:
    """Agent that fetches and extracts structured content from web sources."""

    def __init__(self, relevance_threshold: float = 0.5):
        self.relevance_threshold = relevance_threshold
        self.extracted: list[ExtractedContent] = []

    def fetch_page(self, url: str) -> str:
        """Fetch raw page content. In production: use requests + BeautifulSoup."""
        return f"[Mock page content for {url}] This page discusses important research findings..."

    def extract(self, search_result: SearchResult) -> ExtractedContent:
        """Extract structured information from a single source."""
        page_content = self.fetch_page(search_result.url)
        raw = mock_llm(
            system_prompt="You are a content extractor. Summarize the page and extract key facts.",
            user_prompt=f"Title: {search_result.title}\nContent: {page_content}"
        )
        data = json.loads(raw)
        content = ExtractedContent(
            url=search_result.url,
            title=search_result.title,
            summary=data["summary"],
            key_facts=data["key_facts"],
            relevance_score=data["relevance_score"]
        )
        self.extracted.append(content)
        return content

    def run(self, search_results: list[SearchResult]) -> list[ExtractedContent]:
        """Extract content from all search results, filtering by relevance."""
        extracted = []
        for sr in search_results:
            content = self.extract(sr)
            if content.relevance_score >= self.relevance_threshold:
                extracted.append(content)
                print(f"  [ExtractionAgent] {content.title}: relevance={content.relevance_score:.2f} -> KEEP")
            else:
                print(f"  [ExtractionAgent] {content.title}: relevance={content.relevance_score:.2f} -> SKIP")
        print(f"[ExtractionAgent] {len(extracted)}/{len(search_results)} sources passed relevance filter")
        return extracted

print("ExtractionAgent defined.")

In [ ]:
# --- Test the Extraction Agent ---

extraction_agent = ExtractionAgent(relevance_threshold=0.5)
extracted = extraction_agent.run(results)

for e in extracted:
    print(f"\n  Title: {e.title}")
    print(f"  Summary: {e.summary[:100]}...")
    print(f"  Key facts: {len(e.key_facts)}")

---
## 4 Synthesis Agent

The **Synthesis Agent** combines findings from all extracted sources into a unified narrative. It identifies common themes, contradictions, and knowledge gaps.

In [ ]:
@dataclass
class SynthesisResult:
    """Output of the synthesis agent."""
    narrative: str
    themes: list[str]
    contradictions: list[str]
    knowledge_gaps: list[str]
    source_count: int


class SynthesisAgent:
    """Agent that synthesizes findings from multiple extracted sources."""

    def run(self, extracted_contents: list[ExtractedContent]) -> SynthesisResult:
        """Synthesize all extracted content into a coherent narrative."""
        # Build context from all sources
        source_summaries = []
        for ec in extracted_contents:
            facts_str = "; ".join(ec.key_facts)
            source_summaries.append(f"Source: {ec.title}\nSummary: {ec.summary}\nFacts: {facts_str}")
        combined = "\n\n".join(source_summaries)

        narrative = mock_llm(
            system_prompt="You are a research synthesizer. Combine the findings into a coherent narrative.",
            user_prompt=f"Synthesize these sources:\n\n{combined}"
        )

        result = SynthesisResult(
            narrative=narrative,
            themes=[
                "Task decomposition is critical for multi-agent research quality",
                "Hierarchical supervision reduces errors and improves coordination",
                "Structured citation management enhances verifiability"
            ],
            contradictions=[
                "Sources disagree on whether flat or hierarchical topologies are better for small tasks"
            ],
            knowledge_gaps=[
                "Limited data on cost-effectiveness of multi-agent vs single-agent for simple queries",
                "No consensus on optimal number of agents for different task complexities"
            ],
            source_count=len(extracted_contents)
        )

        print(f"[SynthesisAgent] Synthesized {result.source_count} sources")
        print(f"  Themes: {len(result.themes)} | Contradictions: {len(result.contradictions)} | Gaps: {len(result.knowledge_gaps)}")
        return result

print("SynthesisAgent defined.")

In [ ]:
# --- Test the Synthesis Agent ---

synthesis_agent = SynthesisAgent()
synthesis = synthesis_agent.run(extracted)

print(f"\nNarrative:\n{synthesis.narrative}")
print(f"\nThemes:")
for t in synthesis.themes:
    print(f"  - {t}")
print(f"\nKnowledge Gaps:")
for g in synthesis.knowledge_gaps:
    print(f"  - {g}")

---
## 5 Citation Manager

The **Citation Manager** maintains a structured bibliography, assigns citation keys, and can produce formatted references in multiple styles (APA, MLA, Chicago).

In [ ]:
@dataclass
class Citation:
    """A structured citation entry."""
    key: str
    title: str
    url: str
    accessed_date: str
    summary: str
    relevance_score: float


class CitationManager:
    """Manages citations for the research report."""

    def __init__(self):
        self.citations: dict[str, Citation] = {}
        self._counter = 0

    def _generate_key(self, title: str) -> str:
        """Generate a short citation key from the title."""
        self._counter += 1
        words = title.split()[:3]
        slug = "-".join(w.lower()[:6] for w in words)
        return f"{slug}-{self._counter}"

    def add_source(self, extracted: ExtractedContent) -> str:
        """Add an extracted source to the citation database. Returns the citation key."""
        key = self._generate_key(extracted.title)
        self.citations[key] = Citation(
            key=key,
            title=extracted.title,
            url=extracted.url,
            accessed_date=datetime.now().strftime("%Y-%m-%d"),
            summary=extracted.summary[:200],
            relevance_score=extracted.relevance_score
        )
        return key

    def format_apa(self, key: str) -> str:
        """Format a citation in APA style."""
        c = self.citations[key]
        return f"{c.title}. Retrieved {c.accessed_date}, from {c.url}"

    def format_all(self, style: str = "apa") -> str:
        """Format all citations in the given style."""
        lines = []
        for i, (key, c) in enumerate(self.citations.items(), 1):
            if style == "apa":
                lines.append(f"[{i}] {self.format_apa(key)}")
            else:
                lines.append(f"[{i}] {c.title} ({c.url})")
        return "\n".join(lines)

    def get_inline_ref(self, key: str) -> str:
        """Get an inline citation reference like [1]."""
        keys = list(self.citations.keys())
        idx = keys.index(key) + 1
        return f"[{idx}]"

print("CitationManager defined.")

In [ ]:
# --- Test the Citation Manager ---

citation_mgr = CitationManager()
citation_keys = []
for ec in extracted:
    key = citation_mgr.add_source(ec)
    citation_keys.append(key)
    print(f"  Added: {key} -> {ec.title}")

print(f"\nFormatted References (APA):\n{citation_mgr.format_all('apa')}")

---
## 6 Report Generator

The **Report Generator** takes the synthesis, citations, and metadata and produces a formatted Markdown research report.

In [ ]:
@dataclass
class ResearchReport:
    """A complete research report."""
    title: str
    question: str
    executive_summary: str
    body: str
    themes: list[str]
    knowledge_gaps: list[str]
    references: str
    generated_at: str = field(default_factory=lambda: datetime.now().isoformat())
    source_count: int = 0


class ReportGenerator:
    """Generates a formatted research report from synthesis and citations."""

    def generate(self, question: str, synthesis: SynthesisResult,
                 citation_mgr: CitationManager) -> ResearchReport:
        """Generate a complete research report."""
        # Use LLM to produce the final report body
        report_body = mock_llm(
            system_prompt="You are a report writer. Format the research into a clear report.",
            user_prompt=f"Question: {question}\nNarrative: {synthesis.narrative}"
        )

        themes_md = "\n".join(f"- {t}" for t in synthesis.themes)
        gaps_md = "\n".join(f"- {g}" for g in synthesis.knowledge_gaps)
        refs = citation_mgr.format_all("apa")

        report = ResearchReport(
            title=f"Research Report: {question}",
            question=question,
            executive_summary=synthesis.narrative,
            body=report_body,
            themes=synthesis.themes,
            knowledge_gaps=synthesis.knowledge_gaps,
            references=refs,
            source_count=synthesis.source_count
        )

        print(f"[ReportGenerator] Report generated: {len(report.body)} chars, {report.source_count} sources")
        return report

    def to_markdown(self, report: ResearchReport) -> str:
        """Convert a report to Markdown."""
        themes_md = "\n".join(f"- {t}" for t in report.themes)
        gaps_md = "\n".join(f"- {g}" for g in report.knowledge_gaps)
        return textwrap.dedent(f"""\
        # {report.title}

        **Generated:** {report.generated_at} | **Sources:** {report.source_count}

        ## Executive Summary

        {report.executive_summary}

        ## Key Themes

        {themes_md}

        ## Detailed Findings

        {report.body}

        ## Knowledge Gaps

        {gaps_md}

        ## References

        {report.references}
        """)

print("ReportGenerator defined.")

In [ ]:
# --- Test the Report Generator ---

report_gen = ReportGenerator()
report = report_gen.generate(
    question="What are the latest advances in multi-agent AI systems?",
    synthesis=synthesis,
    citation_mgr=citation_mgr
)

md = report_gen.to_markdown(report)
print(md[:600])
print("\n... [truncated] ...")

---
## 7 Supervisor Orchestration

The **Supervisor** ties everything together. It manages the pipeline state, decides when to move between stages, handles errors, and can request re-runs of any agent.

In [ ]:
class PipelineStage(Enum):
    SEARCH = "search"
    EXTRACT = "extract"
    SYNTHESIZE = "synthesize"
    CITE = "cite"
    REPORT = "report"
    COMPLETE = "complete"


@dataclass
class PipelineState:
    """Tracks the state of the research pipeline."""
    question: str
    stage: PipelineStage = PipelineStage.SEARCH
    search_results: list[SearchResult] = field(default_factory=list)
    extracted_contents: list[ExtractedContent] = field(default_factory=list)
    synthesis: SynthesisResult | None = None
    citation_keys: list[str] = field(default_factory=list)
    report: ResearchReport | None = None
    errors: list[str] = field(default_factory=list)
    started_at: str = field(default_factory=lambda: datetime.now().isoformat())


class Supervisor:
    """Orchestrates the multi-agent research pipeline."""

    def __init__(self):
        self.search_agent = SearchAgent(max_queries=3)
        self.extraction_agent = ExtractionAgent(relevance_threshold=0.5)
        self.synthesis_agent = SynthesisAgent()
        self.citation_mgr = CitationManager()
        self.report_gen = ReportGenerator()

    def run(self, question: str, verbose: bool = True) -> PipelineState:
        """Run the full research pipeline."""
        state = PipelineState(question=question)

        # Stage 1: Search
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 1: SEARCH")
            print(f"{'='*60}")
        try:
            state.search_results = self.search_agent.run(question)
            state.stage = PipelineStage.EXTRACT
        except Exception as e:
            state.errors.append(f"Search failed: {e}")
            return state

        # Stage 2: Extract
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 2: EXTRACT")
            print(f"{'='*60}")
        try:
            state.extracted_contents = self.extraction_agent.run(state.search_results)
            if not state.extracted_contents:
                state.errors.append("No sources passed relevance filter.")
                return state
            state.stage = PipelineStage.SYNTHESIZE
        except Exception as e:
            state.errors.append(f"Extraction failed: {e}")
            return state

        # Stage 3: Cite
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 3: CITATIONS")
            print(f"{'='*60}")
        for ec in state.extracted_contents:
            key = self.citation_mgr.add_source(ec)
            state.citation_keys.append(key)
        if verbose:
            print(f"  [Supervisor] {len(state.citation_keys)} citations registered")
        state.stage = PipelineStage.CITE

        # Stage 4: Synthesize
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 4: SYNTHESIZE")
            print(f"{'='*60}")
        try:
            state.synthesis = self.synthesis_agent.run(state.extracted_contents)
            state.stage = PipelineStage.REPORT
        except Exception as e:
            state.errors.append(f"Synthesis failed: {e}")
            return state

        # Stage 5: Report
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 5: REPORT")
            print(f"{'='*60}")
        try:
            state.report = self.report_gen.generate(question, state.synthesis, self.citation_mgr)
            state.stage = PipelineStage.COMPLETE
        except Exception as e:
            state.errors.append(f"Report generation failed: {e}")
            return state

        if verbose:
            print(f"\n{'='*60}")
            print(f"  PIPELINE COMPLETE")
            print(f"{'='*60}")
            print(f"  Sources found: {len(state.search_results)}")
            print(f"  Sources used:  {len(state.extracted_contents)}")
            print(f"  Citations:     {len(state.citation_keys)}")
            print(f"  Errors:        {len(state.errors)}")

        return state

print("Supervisor defined.")

---
## 8 Full Pipeline Demo

Let us run the entire research pipeline end-to-end.

In [ ]:
# --- Full Pipeline Demo ---

supervisor = Supervisor()
state = supervisor.run("What are the latest advances in multi-agent AI systems?")

In [ ]:
# --- Display the final report ---

if state.report:
    md = supervisor.report_gen.to_markdown(state.report)
    print(md)
else:
    print("Pipeline did not complete. Errors:")
    for err in state.errors:
        print(f"  - {err}")

In [ ]:
# --- Inspect pipeline state ---

print(f"Final stage:  {state.stage.value}")
print(f"Question:     {state.question}")
print(f"Started at:   {state.started_at}")
print(f"Search hits:  {len(state.search_results)}")
print(f"Extracted:    {len(state.extracted_contents)}")
print(f"Citations:    {len(state.citation_keys)}")
print(f"Has synthesis: {state.synthesis is not None}")
print(f"Has report:   {state.report is not None}")
print(f"Errors:       {state.errors}")

---
## 9 Domain Variants

The research assistant architecture generalizes across domains. Below we show how to configure domain-specific variants by adjusting search strategies, extraction prompts, and report templates.

In [ ]:
# --- Domain Configuration ---

@dataclass
class DomainConfig:
    """Configuration for a domain-specific research assistant."""
    name: str
    search_suffixes: list[str]
    extraction_focus: str
    report_sections: list[str]
    relevance_threshold: float

DOMAIN_CONFIGS = {
    "technology": DomainConfig(
        name="Technology Research",
        search_suffixes=["technical paper", "benchmark", "GitHub repository"],
        extraction_focus="Extract technical specifications, performance benchmarks, and code examples.",
        report_sections=["Executive Summary", "Technical Analysis", "Benchmarks", "Implementation Notes", "References"],
        relevance_threshold=0.6
    ),
    "healthcare": DomainConfig(
        name="Healthcare Research",
        search_suffixes=["clinical trial", "PubMed", "systematic review"],
        extraction_focus="Extract study design, sample size, outcomes, and statistical significance.",
        report_sections=["Executive Summary", "Clinical Evidence", "Safety Profile", "Recommendations", "References"],
        relevance_threshold=0.7
    ),
    "finance": DomainConfig(
        name="Finance Research",
        search_suffixes=["SEC filing", "analyst report", "market data"],
        extraction_focus="Extract financial metrics, risk factors, market trends, and regulatory implications.",
        report_sections=["Executive Summary", "Market Analysis", "Risk Assessment", "Outlook", "References"],
        relevance_threshold=0.65
    )
}

for name, cfg in DOMAIN_CONFIGS.items():
    print(f"\n{cfg.name}")
    print(f"  Search suffixes:  {cfg.search_suffixes}")
    print(f"  Report sections:  {cfg.report_sections}")
    print(f"  Relevance cutoff: {cfg.relevance_threshold}")

In [ ]:
# --- Run a domain-specific research pipeline ---

def run_domain_research(question: str, domain: str) -> PipelineState:
    """Run research with domain-specific configuration."""
    cfg = DOMAIN_CONFIGS[domain]
    print(f"Running {cfg.name} pipeline for: {question}")
    print(f"  Config: threshold={cfg.relevance_threshold}, sections={cfg.report_sections}")

    supervisor = Supervisor()
    supervisor.extraction_agent.relevance_threshold = cfg.relevance_threshold
    return supervisor.run(question, verbose=False)

# Example: Healthcare research
health_state = run_domain_research(
    "What are the latest clinical trials for Alzheimer's treatment?",
    domain="healthcare"
)
print(f"\nCompleted: stage={health_state.stage.value}, sources={len(health_state.extracted_contents)}")

---
## 10 Exercises

### Exercise 1 (Conceptual)

The current Supervisor runs agents in a fixed linear sequence: Search -> Extract -> Cite -> Synthesize -> Report.

**Question:** Design a more adaptive supervisor that can:
- Re-run the search if extraction yields too few relevant sources.
- Ask the synthesis agent to flag areas needing more research and trigger additional searches.
- Handle partial failures (e.g., one source times out) without aborting the pipeline.

Draw a state diagram (or describe it in text) showing the transitions between stages, including retry loops and fallback paths.

---

### Exercise 2 (Coding)

Implement a `QualityChecker` agent that evaluates the final report against these criteria:
- Every claim is backed by at least one citation.
- The report addresses all identified themes from the synthesis.
- The executive summary is under 200 words.

The quality checker should return a structured score and a list of improvement suggestions. If the score is below a threshold, the Supervisor should send the report back to the `ReportGenerator` for revision.

```python
@dataclass
class QualityScore:
    citation_coverage: float   # 0.0 - 1.0
    theme_coverage: float      # 0.0 - 1.0
    summary_length_ok: bool
    overall_score: float
    suggestions: list[str]

class QualityChecker:
    def check(self, report: ResearchReport, synthesis: SynthesisResult) -> QualityScore:
        # Your implementation here
        pass
```

---

### Exercise 3 (Design)

Design a **multi-user collaborative research system** where:
- Multiple users can submit research questions that build on each other.
- The system maintains a shared knowledge base across research sessions.
- Previously extracted and synthesized content is reused when relevant.
- Users can annotate, correct, or extend the system's findings.

Write a design document (1-2 pages) covering:
1. System architecture diagram
2. Data model for the shared knowledge base
3. How you would detect relevance between new questions and existing knowledge
4. Concurrency and conflict resolution strategies

In [ ]:
# --- Scratch cell for exercises ---

# Use this cell to work on the exercises above.
print("Ready for exercises.")